# LangChain Agent Benchmark 02: RAG, Embeddings, Chunking, Vector Stores, and Retrieval

This notebook compares retrieval settings using the text from `docs/paper.pdf`. It uses LangChain's document loader, text splitter, embedding, vector store, and retrieval interfaces directly.
Careful: RAG can only be used on unstructured text, ie. papers, documentation, etc., not tables or databases. RAG adaptations to tabular data exist but are not state of the art RAG. Reliable methods to query structured text follow in notebook 03.


In [ ]:
# Run once per environment. Keep optional provider packages commented until needed.
%pip install -qU langchain langchain-core langchain-community langchain-openai langchain-anthropic langchain-google-genai langchain-text-splitters langgraph pandas pydantic pypdf langchain-huggingface sentence-transformers
# Optional local/vector-store packages:
# %pip install -qU faiss-cpu langchain-chroma langchain-qdrant qdrant-client


In [ ]:
import os
import time
from pathlib import Path

import pandas as pd

from langchain.chat_models import init_chat_model
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

if not os.getenv("GOOGLE_API_KEY"):
    print("Set GOOGLE_API_KEY before running the examples.")

PDF_PATH = Path("docs/paper.pdf")
bio_docs = PyPDFLoader(str(PDF_PATH)).load()

for doc in bio_docs:
    doc.metadata["source"] = str(PDF_PATH)
    doc.metadata["doc_type"] = "paper_pdf"
    doc.metadata["paper"] = PDF_PATH.name

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)


## 1. Retrieval (RAG)

Retrieval controls whether the answer is generated from model priors alone, retrieved source context, reranked context, or multiple retrieved chunks.
The parameter k controls the top k number of similar chunks retrieved. Less chunks give the mdoel less context so answers are focused and cheaper, more chunks have a better chance of finding the answer but may be dispersive, longer, and more expensive.

In [ ]:
question = "What did dexamethasone treatment do to CRISPLD2 expression in airway smooth muscle cells?"
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

response = model.invoke(question)
no_retrieved_answer = response.content

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)

retrieved_docs = vector_store.similarity_search(question, k=2)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)
prompt = f"Answer using only this context. If the answer is absent, say you do not know.\n\n{context}\n\nQuestion: {question}"
response = model.invoke(prompt)
retrieved_answer = response.content

more_docs = vector_store.similarity_search(question, k=6)
context = "\n\n".join(doc.page_content for doc in more_docs)
prompt = f"Answer using only this context. If the answer is absent, say you do not know.\n\n{context}\n\nQuestion: {question}"
response = model.invoke(prompt)
more_context_answer = response.content

df = pd.DataFrame([
    {"mode": "no_rag", "answer": no_retrieved_answer, "sources": ""},
    {"mode": "rag_k_2", "answer": retrieved_answer, "sources": [doc.metadata for doc in retrieved_docs]},
    {"mode": "rag_k_6", "answer": more_context_answer, "sources": [doc.metadata for doc in more_docs]},
])
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 2. Embedding model

Embedding model controls how documents and queries are represented for semantic search, which affects retrieval precision and relevance. The models below differ by size, training objective, language coverage, and domain specialization.


In [ ]:
query = "CRISPLD2 glucocorticoid dexamethasone airway smooth muscle"
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)

from langchain_huggingface import HuggingFaceEmbeddings

embedding_models = [
    {
        "label": "MiniLM general small",
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "why_different": "Small, fast, 384-dim general sentence encoder.",
    },
    # {
    #     "label": "MPNet general larger",
    #     "model_name": "sentence-transformers/all-mpnet-base-v2",
    #     "why_different": "Larger 768-dim general encoder; often stronger but slower.",
    # },
    {
        "label": "Multi-QA retrieval tuned",
        "model_name": "sentence-transformers/multi-qa-mpnet-base-dot-v1",
        "why_different": "Trained on question-answer pairs for semantic search.",
    },
    # {
    #     "label": "Multilingual paraphrase",
    #     "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    #     "why_different": "Multilingual 384-dim model; trades English-only focus for language coverage.",
    # },
    {
        "label": "PubMedBERT biomedical",
        "model_name": "NeuML/pubmedbert-base-embeddings",
        "why_different": "Biomedical literature model tuned on PubMed title-abstract pairs.",
    },
]

rows = []
for spec in embedding_models:
    try:
        start = time.perf_counter()
        embeddings = HuggingFaceEmbeddings(
            model_name=spec["model_name"],
            encode_kwargs={"normalize_embeddings": True},
        )
        dimension = len(embeddings.embed_query("dimension check"))
        vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
        hits = vector_store.similarity_search(query, k=3)
        rows.append({
            "embedding": spec["label"],
            "model_name": spec["model_name"],
            "why_different": spec["why_different"],
            "dimension": dimension,
            "seconds": round(time.perf_counter() - start, 2),
            "top_hits": [doc.page_content[:140] for doc in hits],
            "metadata": [doc.metadata for doc in hits],
        })
    except Exception as exc:
        rows.append({
            "embedding": spec["label"],
            "model_name": spec["model_name"],
            "why_different": spec["why_different"],
            "dimension": None,
            "seconds": None,
            "top_hits": [],
            "metadata": [],
            "error": repr(exc),
        })

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_hits"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)



## 3. Chunk size

Chunk size controls the amount of text in each indexed unit, balancing complete context against retrieval specificity.


In [ ]:
query = "What RNA-Seq quality control or alignment metrics are reported?"
rows = []

for size in [200, 500, 1000, 2000]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=100)
    splits = text_splitter.split_documents(bio_docs)
    vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"chunk_size": size, "n_chunks": len(splits), "top_context": [doc.page_content for doc in hits]})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 4. Chunk overlap

Chunk overlap controls how much text is repeated across adjacent chunks, reducing missing context at boundaries but potentially increasing duplicate retrieval.


In [ ]:
query = "What treatment protocol was used for dexamethasone in airway smooth muscle cells?"
rows = []

for overlap in [0, 100, 300]:
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=overlap)
    splits = text_splitter.split_documents(bio_docs)
    vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
    hits = vector_store.similarity_search(query, k=4)
    rows.append({"chunk_overlap": overlap, "n_chunks": len(splits), "top_context": [doc.page_content for doc in hits]})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_context"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 5. Vector store

Vector store controls where embeddings are stored and queried, affecting speed, usability, persistence, and scalability.


In [ ]:
query = "CRISPLD2 dexamethasone cytokine IL6 IL8"
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
rows = []

start = time.perf_counter()
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
hits = vector_store.similarity_search(query, k=3)
rows.append({"store": "InMemoryVectorStore", "seconds": round(time.perf_counter() - start, 4), "top_metadata": [doc.metadata for doc in hits]})

try:
    from langchain_chroma import Chroma

    start = time.perf_counter()
    vector_store = Chroma.from_documents(documents=splits, embedding=embeddings, collection_name="bio_benchmark")
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"store": "Chroma", "seconds": round(time.perf_counter() - start, 4), "top_metadata": [doc.metadata for doc in hits]})
except Exception as exc:
    print("Chroma skipped:", exc)

try:
    from langchain_community.vectorstores import FAISS

    start = time.perf_counter()
    vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
    hits = vector_store.similarity_search(query, k=3)
    rows.append({"store": "FAISS", "seconds": round(time.perf_counter() - start, 4), "top_metadata": [doc.metadata for doc in hits]})
except Exception as exc:
    print("FAISS skipped:", exc)

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["top_metadata"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 6. Similarity search

Similarity search retrieves chunks closest to the query embedding, usually favoring relevance over diversity.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)

vector_store.similarity_search("CRISPLD2 IL1 beta IL6 IL8 knockdown", k=3)


## 7. MMR retrieval

MMR retrieval trades off similarity with diversity, which can reduce redundant chunks when several sources say similar things.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)

vector_store.max_marginal_relevance_search("RNA-Seq transcriptome profiling dexamethasone glucocorticoid response", k=3)


## 8. Metadata filtering

Metadata filtering restricts retrieval to documents with selected labels, such as source file or PDF page ranges.


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)

first_page_hits = [
    doc for doc in vector_store.similarity_search("abstract CRISPLD2 dexamethasone", k=8)
    if doc.metadata.get("page") == 0
]
early_results_hits = [
    doc for doc in vector_store.similarity_search("RNA-Seq results differentially expressed genes", k=8)
    if 1 <= doc.metadata.get("page", -1) <= 4
]
paper_pdf_hits = [
    doc for doc in vector_store.similarity_search("glucocorticoid airway smooth muscle", k=8)
    if doc.metadata.get("doc_type") == "paper_pdf"
]

df = pd.DataFrame([
    {"filter": "first PDF page", "hits": [doc.metadata for doc in first_page_hits]},
    {"filter": "early results pages", "hits": [doc.metadata for doc in early_results_hits]},
    {"filter": "paper PDF source", "hits": [doc.metadata for doc in paper_pdf_hits]},
])
df.style.set_properties(
    subset=["hits"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)


## 9. Top-k retrieval

Top-k retrieval controls how many chunks are passed to the generator, affecting evidence coverage, distraction, token cost, and answer length.


In [ ]:
query = "What were the main RNA-Seq findings about CRISPLD2 and cytokine regulation?"
model = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(bio_docs)
vector_store = InMemoryVectorStore.from_documents(documents=splits, embedding=embeddings)
rows = []

for k in [2, 5, 10]:
    retrieved_docs = vector_store.similarity_search(query, k=k)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = f"Answer using only this context. If the answer is absent, say you do not know.\n\n{context}\n\nQuestion: {query}"
    response = model.invoke(prompt)
    rows.append({"k": k, "n_sources": len(retrieved_docs), "sources": [doc.metadata for doc in retrieved_docs], "answer": response.content})

df = pd.DataFrame(rows)
df.style.set_properties(
    subset=["answer"],
    **{"white-space": "pre-wrap", "text-align": "left", "min-width": "650px"},
)
